# Practical 02 — Multitemporal Vegetation Dynamics & Crop Phenology

**Platform:** Google Colab + each student's Google Earth Engine project + their own Google Drive
**Environment:** Colab-safe v4 (enhanced ROI capture + sequential, checkpointed Earth Engine statistics); no forced pip upgrades or authentication-library pins  
**Primary data:** Harmonized Sentinel-2 Level-2A Surface Reflectance (10 m)  
**Study:** Three student-drawn agricultural fields, **May 1–October 31**, 2023–2025 (editable)  
**Methods:** NDVI & EVI; 10-day median composites; short-gap interpolation; Savitzky–Golay smoothing; satellite-derived phenological metrics; interannual comparison; maps.

### Learning objectives
1. Define and validate three individual agricultural field polygons.
2. Generate reproducible Sentinel-2 NDVI/EVI time series using quality masks.
3. Quantify the effect of compositing, missing observations and temporal smoothing.
4. Estimate **SOS**, **POS**, **EOS**, amplitude and seasonal NDVI integrals.
5. Evaluate **20% versus 30%** relative-amplitude phenology thresholds.
6. Interpret variability between fields and years, without assuming that NDVI dates are ground-observed crop stages.

> **Important:** SOS/EOS are satellite greenness indicators, **not** planting, emergence, flowering, or harvest dates. This notebook does not identify crop type. All figures and tables are stored on the **student's own Google Drive**. Earth Engine GeoTIFFs are exported using separate asynchronous tasks.

### Workflow
Draw three field polygons → process Sentinel-2 → 10-day median index composites → field means → quality checking and smoothing → phenology → maps → export and interpretation.

## 0. Use Colab's preinstalled Python environment — **do not upgrade packages**

Google Colab already includes the Earth Engine Python API, geemap, pandas, NumPy, SciPy, Matplotlib, Shapely and Pillow in standard runtimes. **Do not use** `%pip install -U earthengine-api geemap ...` or pin `google-auth`, `pandas` or `Pillow`; these commands can cause version conflicts with Colab.

**If you previously ran a package-upgrade cell and got dependency conflicts:** in Colab, choose **Runtime → Disconnect and delete runtime**, reconnect, and run this notebook again **without** the old installation command. This is more reliable than merely restarting the Python kernel, since restart alone does not undo pip changes.

The next cell **checks** installed packages but does not change them. If a fresh Colab image has missing packages, ask your instructor for a tested environment rather than bulk-upgrading. A Google authentication and Earth Engine version mismatch should be fixed as a coherent set, not by changing `google-auth` alone.

In [ ]:
# Google Colab generally provides everything this practical needs.
# This cell makes NO pip changes to the Colab environment.
from importlib import metadata, util

packages = {
    'ee': 'earthengine-api',
    'geemap': 'geemap',
    'ipywidgets': 'ipywidgets',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'shapely': 'shapely',
    'matplotlib': 'matplotlib',
    'PIL': 'pillow',
}
missing = [distribution for module, distribution in packages.items()
           if util.find_spec(module) is None]
if missing:
    raise RuntimeError(
        'Required packages missing from this Colab runtime: ' + ', '.join(missing) +
        '. Start with a fresh standard Google Colab runtime. '
        'Do not bulk-upgrade Earth Engine or google-auth.'
    )

for distribution in [*packages.values(), 'google-auth', 'google-colab']:
    try:
        print(f'{distribution}: {metadata.version(distribution)}')
    except metadata.PackageNotFoundError:
        print(f'{distribution}: not installed')

# Detect (without modifying) the EE/google-auth compatibility mismatch.
try:
    from packaging.requirements import Requirement
    from packaging.version import Version
    google_auth_ver = Version(metadata.version('google-auth'))
    ee_reqs = metadata.requires('earthengine-api') or []
    for req_string in ee_reqs:
        req = Requirement(req_string)
        if req.name.lower().replace('_', '-') == 'google-auth' and (
                req.marker is None or req.marker.evaluate()):
            if google_auth_ver not in req.specifier:
                print('\nWARNING: Installed earthengine-api requires google-auth',
                      str(req.specifier), 'but found', google_auth_ver)
                print('If you previously ran pip upgrades, select '
                      'Runtime > Disconnect and delete runtime, and run again.')
except (ImportError, metadata.PackageNotFoundError) as exc:
    print('Could not complete optional dependency check:', exc)

print('\nSetup complete; no packages were installed or upgraded.')

In [ ]:
import os, re, json, warnings, urllib.request
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import dates as mdates
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LinearSegmentedColormap
from scipy.signal import savgol_filter
from shapely.geometry import shape, mapping
from IPython.display import display
import ee
import geemap

# Enable two-way Colab widgets for the field drawing map, when available.
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except (ImportError, AttributeError):
    pass

plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 300, 'font.size': 10,
    'axes.grid': True, 'grid.alpha': 0.18, 'axes.spines.top': True,
    'axes.spines.right': True
})
print('Imports successful.')

## 1. Student configuration and Earth Engine authentication

Use **your own registered Google Earth Engine cloud project** and your own student identifier. Run the authentication interactively; no credentials or tokens are written to the notebook. Never share the resulting private Drive files without permission.

In [ ]:
#@title Practical 2 — Student configuration
# ===== EACH STUDENT MUST EDIT THESE TWO ENTRIES =====
EE_PROJECT_ID = 'CHANGE_ME'  #@param {type:"string"} # Your Earth Engine-enabled project
STUDENT_ID = 'FirstName_LastName'  #@param {type:"string"} # Your own Drive output folder

# ===== EXPERIMENT CONFIGURATION =====
YEARS = [2023, 2024, 2025]           # select complete years with Sentinel-2 observations
START_MONTH_DAY = (5, 1)
END_MONTH_DAY_EXCLUSIVE = (11, 1)     # includes all dates through October 31
COMPOSITE_DAYS = 10
CLOUD_SCORE_THRESHOLD = 0.60         # Cloud Score+ cs_cdf threshold
SCENE_CLOUD_LIMIT = 90               # scene-level screening; pixel QA is primary
FIELD_SCALE_M = 10                   # field statistics at 10-m nominal resolution

# Smoothing/phenology settings
MAX_INTERPOLATED_GAP_BINS = 2         # <=20 days missing internal bins
SAVGOL_WINDOW_BINS = 5               # must be odd; 5*10-day bins = ~50-day window
SAVGOL_POLY_ORDER = 2
MIN_VALID_BINS = 10
MIN_SEASONAL_AMPLITUDE = 0.15         # reject very weak NDVI seasons
PHENOLOGY_THRESHOLDS = [0.20, 0.30]
MIN_PIXEL_COMPOSITES = 6             # threshold for reliable pixel quicklooks

# Optional tasks. Run the last export section after reviewing field curves.
START_GEOTIFF_EXPORTS = True
MAKE_MAP_QUICKLOOKS = True           # PNG thumbnails; may be blocked by proxy/permissions

if EE_PROJECT_ID == 'CHANGE_ME' or not EE_PROJECT_ID.strip():
    raise ValueError('Edit EE_PROJECT_ID to your own registered Earth Engine Cloud project.')
if STUDENT_ID == 'FirstName_LastName':
    raise ValueError('Edit STUDENT_ID to your own name/identifier.')
if not (SAVGOL_WINDOW_BINS % 2 == 1 and SAVGOL_WINDOW_BINS > SAVGOL_POLY_ORDER):
    raise ValueError('Savitzky-Golay window must be odd and exceed polynomial order.')
if len(YEARS) < 2 or len(set(YEARS)) != len(YEARS):
    raise ValueError('Choose at least two distinct years for interannual comparison.')
if max(YEARS) >= datetime.now().year and datetime.now() < datetime(max(YEARS), 11, 1):
    raise ValueError('The most recent selected year does not yet have a complete May–October season.')

TAG = re.sub(r'[^A-Za-z0-9_-]+', '_', STUDENT_ID).strip('_')[:50]
try:
    ee.Initialize(project=EE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_ID)
print('Earth Engine initialized using:', EE_PROJECT_ID)
print('Student:', TAG, 'Years:', YEARS)

## 2. Mount **your own** Google Drive and create output folders

Google Colab must be signed in to the Google account whose Drive should receive the results. The Google Earth Engine project and Google Drive account may be different, but each must be authorized individually.

Outputs written via Python go inside `MyDrive/Satellite_Practicals/Practical_02/<student>/`. **Earth Engine exports cannot target a nested Drive path reliably**; they go to a separate top-level folder named `P02_<student>_GeoTIFF`.

In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('This notebook is designed to run inside Google Colab.') from exc

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Satellite_Practicals/Practical_02') / TAG
DIRS = {k: ROOT/k for k in ['ROI', 'Tables', 'Figures', 'Map_Quicklooks', 'Logs']}
for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)
EXPORT_DRIVE_FOLDER = f'P02_{TAG}_GeoTIFF'  # distinct top-level Drive folder
print('Notebook output folder:', ROOT)
print('Earth Engine GeoTIFF task destination: My Drive /', EXPORT_DRIVE_FOLDER)

## 3. Select **three separate agricultural field polygons** interactively

1. Run the map cell **once**, and navigate to the region you want to study.
2. Using the polygon or rectangle tool in the map toolbar, **draw one outline for Field A, one for Field B, and one for Field C**. Finish each outline before starting the next. **One large polygon containing three fields counts as only one field**.
3. Use the small **Check field count** cell immediately below the map. It should report **3 / 3**. Drawing zero or one polygon is not sufficient for the three-field comparison.
4. When the count reaches 3, run the **Capture and validate fields** cell. It saves the three field boundaries in **your own Google Drive**.
5. **Do not re-run the map creation cell after drawing**: creating a new `roi_map` can erase the polygons already drawn in the old map. If you need to add more, draw them on the *existing* map and re-run only the count/capture cells.
6. If the widget does not record the shapes, use `ROI_INPUT_MODE='geojson'` with **three individual polygon features**, or `ROI_INPUT_MODE='manual_boxes'` with three field rectangles. Do not use a `FeatureCollection` containing only one outer ROI.

**Basemap:** Esri satellite imagery is used instead of OpenStreetMap to avoid the tile-server 403 reported earlier.


In [ ]:
ROI_INPUT_MODE = 'draw'   # 'draw', 'geojson', or 'manual_boxes'
GEOJSON_PATH = '/content/three_fields.geojson'  # when using 'geojson'

# Fallback ONLY: replace with your own [west, south, east, north] field rectangles.
# This example intentionally contains no real coordinates. Never leave empty when selecting manual_boxes.
MANUAL_FIELD_BBOXES = []

roi_map = None
if ROI_INPUT_MODE == 'draw':
    try:
        roi_map = geemap.Map(center=[49.60, -98.18], zoom=12, basemap='Esri.WorldImagery',
                             draw_ctrl=True, measure_ctrl=True)
        display(roi_map)
        print('Draw 3 separate field outlines, finish each shape, then run Check field count.')
    except Exception as exc:
        print('Interactive map unavailable:', exc)
        print("Set ROI_INPUT_MODE='geojson' or 'manual_boxes' and rerun the cell.")

### Check field count (repeat after drawing)
This cell **reads** the drawings but does not replace the map. If it reports fewer than three polygons, return to the map immediately above and draw the remaining fields.

In [ ]:
# Check drawings WITHOUT creating a new map or removing existing fields.
# Run this as many times as necessary after adding shapes to the map above.
if ROI_INPUT_MODE == 'draw':
    if roi_map is None:
        print('The interactive map is unavailable. Choose the GeoJSON/manual-box fallback.')
    else:
        rois = getattr(roi_map, 'user_rois', None)
        if rois is not None:
            count = int(ee.FeatureCollection(rois).size().getInfo())
        else:
            control = getattr(roi_map, 'draw_control', None)
            if control is None:
                control = getattr(roi_map, '_draw_control', None)
            collection = getattr(control, 'collection', None) if control is not None else None
            count = int(ee.FeatureCollection(collection).size().getInfo()) if collection is not None else len(getattr(roi_map, 'draw_features', []) or [])
        print(f'Field polygons captured: {count} / 3')
        if count < 3:
            print(f'Draw {3-count} more SEPARATE field outline(s) on the existing map.')
            print('Do NOT re-run the map creation cell; it would reset the drawings.')
        elif count > 3:
            print('More than three shapes detected; delete the unwanted drawing(s).')
        else:
            print('Good: 3 individual fields recorded. Proceed to the capture cell.')
else:
    print('Draw-count check is for interactive mode. Use capture cell to validate uploaded/manual fields.')


In [ ]:
def extract_polygons_from_geojson(data):
    """Read one or more distinct polygon geometries from GeoJSON/EE responses.

    An outer ROI containing multiple fields is not automatically split into
    three fields; each field should be drawn as a separate map feature.
    """
    if not data or not isinstance(data, dict):
        return []
    kind = data.get('type')
    if kind == 'FeatureCollection':
        result = []
        for feature in data.get('features', []):
            result.extend(extract_polygons_from_geojson(feature))
        return result
    if kind == 'Feature':
        return extract_polygons_from_geojson(data.get('geometry'))
    if kind == 'GeometryCollection':
        result = []
        for geometry in data.get('geometries', []):
            result.extend(extract_polygons_from_geojson(geometry))
        return result
    if kind in ('Polygon', 'MultiPolygon'):
        return [data]
    return []


def get_drawn_geometries(map_widget):
    """Read all drawn features; supports several geemap drawing interfaces."""
    candidates = []

    # Modern geemap: user_rois is an ee.FeatureCollection of *all* drawings.
    rois = getattr(map_widget, 'user_rois', None)
    if rois is not None:
        try:
            candidates.append(extract_polygons_from_geojson(
                ee.FeatureCollection(rois).getInfo()))
        except Exception as exc:
            print('Could not read user_rois:', exc)

    # Some geemap versions expose the draw control's collection.
    control = getattr(map_widget, 'draw_control', None)
    if control is None:
        control = getattr(map_widget, '_draw_control', None)
    collection = getattr(control, 'collection', None) if control is not None else None
    if collection is not None:
        try:
            candidates.append(extract_polygons_from_geojson(
                ee.FeatureCollection(collection).getInfo()))
        except Exception as exc:
            print('Could not read draw-control collection:', exc)

    # Alternate interface: drawn features may be GeoJSON dictionaries or EE features.
    try:
        drawn = getattr(map_widget, 'draw_features', None)
        if drawn is not None:
            collected = []
            for item in drawn:
                if isinstance(item, dict):
                    collected.extend(extract_polygons_from_geojson(item))
                elif hasattr(item, 'geometry'):
                    collected.extend(extract_polygons_from_geojson(
                        ee.Feature(item).geometry().getInfo()))
            candidates.append(collected)
    except Exception as exc:
        print('Could not read draw_features:', exc)

    # Choose the most complete of the available interfaces; do not combine
    # their outputs because the same polygons may appear in multiple sources.
    return max(candidates, key=len) if candidates else []


def bbox_as_polygon(box):
    w, s, e, n = map(float, box)
    if not (w < e and s < n):
        raise ValueError('Each manual bounding box requires west < east and south < north.')
    return {'type': 'Polygon', 'coordinates': [[[w,s],[e,s],[e,n],[w,n],[w,s]]]}

if ROI_INPUT_MODE == 'draw':
    if roi_map is None:
        raise RuntimeError('The map widget is missing. Use the GeoJSON or manual_boxes fallback.')
    selected_geometries = get_drawn_geometries(roi_map)
elif ROI_INPUT_MODE == 'geojson':
    with open(GEOJSON_PATH, 'r', encoding='utf-8') as f:
        selected_geometries = extract_polygons_from_geojson(json.load(f))
elif ROI_INPUT_MODE == 'manual_boxes':
    selected_geometries = [bbox_as_polygon(b) for b in MANUAL_FIELD_BBOXES]
else:
    raise ValueError('ROI_INPUT_MODE must be draw, geojson, or manual_boxes.')

print(f'Individual field polygons detected: {len(selected_geometries)} / 3')
if len(selected_geometries) != 3:
    if ROI_INPUT_MODE == 'draw':
        raise ValueError(
            f'Captured {len(selected_geometries)} field polygon(s), but Practical 2 requires 3. '
            'Draw each field SEPARATELY on the EXISTING map, then rerun only the '
            'count and capture cells. Do NOT rerun the map creation cell.')
    raise ValueError(
        f'Captured {len(selected_geometries)} polygon feature(s); supply exactly '
        '3 separate agricultural field polygons, not one outer study area.')

validated = []
for i, geom in enumerate(selected_geometries, start=1):
    poly = shape(geom)
    if poly.geom_type not in ('Polygon', 'MultiPolygon'):
        raise ValueError(f'Field {i} must be a polygon, not a point or line.')
    if poly.is_empty or not poly.is_valid:
        raise ValueError(f'Field {i} is invalid/self-intersecting. Redraw a simple polygon.')
    # Check realistic field size in square metres with EE geodesic geometry.
    geometry = ee.Geometry(mapping(poly))
    area_ha = float(geometry.area(maxError=1).divide(10000).getInfo())
    if area_ha < 0.5 or area_ha > 1000:
        raise ValueError(f'Field {i} is {area_ha:.2f} ha. Choose a field between 0.5 and 1000 ha.')
    validated.append((mapping(poly), geometry, area_ha))

field_features = []
geojson_features = []
for idx, (geometry_json, geom_ee, area_ha) in enumerate(validated):
    field_name = f'Field_{chr(65 + idx)}'
    field_features.append(ee.Feature(geom_ee, {'field_id': field_name, 'area_ha': area_ha}))
    geojson_features.append({'type':'Feature', 'properties':{'field_id':field_name,'area_ha':round(area_ha,3)},
                             'geometry': geometry_json})

FIELDS_EE = ee.FeatureCollection(field_features)
ROI_EE = FIELDS_EE.geometry(maxError=1)
GEOJSON_FILE = DIRS['ROI']/'selected_fields.geojson'
with open(GEOJSON_FILE, 'w', encoding='utf-8') as f:
    json.dump({'type':'FeatureCollection','features':geojson_features}, f, indent=2)
print('Saved field boundaries:', GEOJSON_FILE)
print('Field areas:', [(f['properties']['field_id'], f['properties']['area_ha']) for f in geojson_features])

## 4. Sentinel-2 surface reflectance, Cloud Score+, and vegetation indices

**Source:** `COPERNICUS/S2_SR_HARMONIZED` (B2 blue, B4 red, B8 near-infrared; all 10 m) with reflectance scaling × 0.0001. We link `GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED` by matching source images and require `cs_cdf ≥ 0.60`. We additionally exclude Scene Classification Layer classes for cloud/shadow/snow/no-data. Clear SCL classes retained: vegetation (4), non-vegetated (5), water (6), unclassified (7). Cloud Score+ is still the primary per-pixel cloud screening.

\[NDVI = \frac{NIR-Red}{NIR+Red}\]

\[EVI = 2.5\,\frac{NIR-Red}{NIR+6Red-7.5Blue+1}\]

Indices are calculated **per valid satellite scene**; each 10-day bin is then the **per-pixel median index** from valid observations in that interval. Empty time bins are preserved as missing rather than manufactured observations.

In [ ]:
# Keep S2 scene-level filter permissive: individual field pixels use Cloud Score+ and SCL.
cs_collection = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
scene_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(ROI_EE)
    .filterDate(f'{min(YEARS)}-05-01', f'{max(YEARS)}-11-01')
    .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', SCENE_CLOUD_LIMIT)))
linked_collection = scene_collection.linkCollection(cs_collection, ['cs_cdf'])


def prepare_sentinel2(source_image):
    img = ee.Image(source_image)
    scl = img.select('SCL')
    land_ok = (scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7)))
    clear = img.select('cs_cdf').gte(CLOUD_SCORE_THRESHOLD).And(land_ok)
    blue = img.select('B2').multiply(0.0001)
    red  = img.select('B4').multiply(0.0001)
    nir  = img.select('B8').multiply(0.0001)

    ndvi_denom = nir.add(red)
    ndvi = nir.subtract(red).divide(ndvi_denom).rename('NDVI')
    ndvi = ndvi.updateMask(ndvi_denom.abs().gt(0.0001))

    evi_denom = nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1)
    evi = nir.subtract(red).multiply(2.5).divide(evi_denom).rename('EVI')
    evi = evi.updateMask(evi_denom.abs().gt(0.1))

    out = ndvi.addBands(evi).updateMask(clear)
    return ee.Image(out.copyProperties(img, ['system:time_start']))

s2_indices = linked_collection.map(prepare_sentinel2)
scene_counts = {year: scene_collection.filterDate(f'{year}-05-01',f'{year}-11-01').size()
                for year in YEARS}
scene_count_info = ee.Dictionary({str(y): n for y, n in scene_counts.items()}).getInfo()
print('Source scenes overlapping selected fields (before pixel QA):', scene_count_info)
if not any(int(n)>0 for n in scene_count_info.values()):
    raise RuntimeError('No Sentinel-2 scenes found. Check field coordinates and years.')

## 5. Create consecutive 10-day median composites for each year

Dates are generated on the **client**, but satellite image operations execute on Earth Engine. Composite start is inclusive and end exclusive. The last interval may contain fewer than 10 days. Each image has an independent midpoint date, day-of-year and count of source scenes.

In [ ]:
YEARS_COLLECTIONS = {}
YEARS_IMAGES = {}
BIN_METADATA = []

for year in YEARS:
    season_start = datetime(year, *START_MONTH_DAY)
    season_end = datetime(year, *END_MONTH_DAY_EXCLUSIVE)
    step = timedelta(days=COMPOSITE_DAYS)
    bin_images = []
    start = season_start
    while start < season_end:
        end = min(start + step, season_end)
        midpoint = start + (end-start)/2
        num = s2_indices.filterDate(start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d')).size()
        selected = s2_indices.filterDate(start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d'))
        # Consistently named, fully masked image when the window has no scenes.
        blank = ee.Image.constant([0, 0]).rename(['NDVI','EVI']).updateMask(ee.Image.constant(0))
        composite = ee.Image(ee.Algorithms.If(num.gt(0), selected.median().select(['NDVI','EVI']), blank))
        doy = (midpoint - datetime(year,1,1)).total_seconds()/86400 + 1
        doy_band = ee.Image.constant(doy).toFloat().rename('DOY').updateMask(composite.select('NDVI').mask())
        composite = composite.addBands(doy_band).set({
            'year':year, 'bin_start':start.strftime('%Y-%m-%d'),
            'bin_end_exclusive':end.strftime('%Y-%m-%d'),
            'bin_midpoint':midpoint.strftime('%Y-%m-%d'),
            'mid_doy':float(doy), 'scene_count':num,
            'system:time_start':ee.Date(midpoint.strftime('%Y-%m-%d')).millis()
        })
        bin_images.append(composite)
        BIN_METADATA.append({'year':year,'bin_start':start.strftime('%Y-%m-%d'),
                             'bin_end_exclusive':end.strftime('%Y-%m-%d'),
                             'bin_midpoint':midpoint.strftime('%Y-%m-%d'), 'mid_doy':doy})
        start = end
    YEARS_IMAGES[year] = bin_images
    YEARS_COLLECTIONS[year] = ee.ImageCollection.fromImages(bin_images)
    print(f'{year}: {len(bin_images)} 10-day windows')

pd.DataFrame(BIN_METADATA).to_csv(DIRS['Tables']/'composite_calendar.csv',index=False)

## 6. Extract field statistics — **sequential, restartable Earth Engine requests**

We avoid building a large feature collection containing **all 19 time-bin × 3 field reductions** in a single `getInfo()` request. That pattern can trigger Earth Engine's **“Too many concurrent aggregations”** (HTTP 429) limit.

Instead, the notebook evaluates **one 10-day composite at a time**, extracting the NDVI/EVI mean and valid-pixel count for the three fields. If a small multi-field request still encounters that limit, it automatically tries **one field at a time**. This avoids merely repeating the same oversized request.

**Progress is saved to your Google Drive after every successful 10-day bin** in `Tables/time_bin_extraction_checkpoint.csv`. If a request fails, re-run **only this cell**: completed bins are skipped. The checkpoint is tied to the study-field geometries, selected years, sensor and processing settings, so results for a different ROI are not mixed together. This is meant for interactive field-sized ROIs; large regions may require a batch table export.

In [ ]:
# Concurrency-safe version: evaluate one composite at a time, save every bin.
# No nested reductions over the whole multi-year series inside one getInfo().
import hashlib
import time

CHECKPOINT_PATH = DIRS['Tables'] / 'time_bin_extraction_checkpoint.csv'
CHECKPOINT_META = DIRS['Tables'] / 'time_bin_extraction_checkpoint_meta.json'

# Detect a changed ROI/settings when a student re-runs with a new configuration.
checkpoint_settings = {
    'fields': geojson_features,
    'years': [int(y) for y in YEARS],
    'composite_days': COMPOSITE_DAYS,
    'period': [START_MONTH_DAY, END_MONTH_DAY_EXCLUSIVE],
    'field_scale_m': FIELD_SCALE_M,
    'cloud_score_threshold': CLOUD_SCORE_THRESHOLD,
    'scene_cloud_limit': SCENE_CLOUD_LIMIT,
    'collection': 'COPERNICUS/S2_SR_HARMONIZED',
    'version': 'P02_v4_sequential',
}
checkpoint_hash = hashlib.sha256(
    json.dumps(checkpoint_settings, sort_keys=True).encode('utf-8')
).hexdigest()

if CHECKPOINT_PATH.exists():
    if not CHECKPOINT_META.exists():
        raise RuntimeError(
            f'Checkpoint {CHECKPOINT_PATH} exists but its metadata is missing. '
            'Rename the old checkpoint before processing a potentially new ROI.'
        )
    with open(CHECKPOINT_META, encoding='utf-8') as f:
        saved_hash = json.load(f).get('signature')
    if saved_hash != checkpoint_hash:
        raise RuntimeError(
            'Existing checkpoint was created using a different ROI or processing settings. '
            f'For a new analysis, move/rename {CHECKPOINT_PATH.name} and '
            f'{CHECKPOINT_META.name} (in {DIRS["Tables"]}) before rerunning.'
        )
    existing_df = pd.read_csv(CHECKPOINT_PATH)
    all_records = existing_df.to_dict('records')
    print(f'Resuming: loaded {len(all_records)} previously saved field records.')
else:
    all_records = []
    with open(CHECKPOINT_META, 'w', encoding='utf-8') as f:
        json.dump({'signature': checkpoint_hash, 'settings': checkpoint_settings},
                  f, indent=2)

reducer = ee.Reducer.mean().combine(
    reducer2=ee.Reducer.count(), sharedInputs=True)
EXPECTED_IDS = {'Field_A', 'Field_B', 'Field_C'}


def persist_checkpoint(rows):
    """Persist progress after each successful time bin; reruns are idempotent."""
    pd.DataFrame(rows).to_csv(CHECKPOINT_PATH, index=False)


def get_completed_bins(rows):
    """Only bins with all three distinct field IDs count as completed."""
    by_bin = {}
    for r in rows:
        try:
            key = (int(r['year']), str(r['bin_start'])[:10])
            by_bin.setdefault(key, set()).add(str(r['field_id']))
        except (KeyError, TypeError, ValueError):
            continue
    return {key for key, ids in by_bin.items() if ids == EXPECTED_IDS}


def fetch_bin_three_fields(img, metadata):
    """One small EE request containing only the three field reductions."""
    z = ee.Image(img)
    sampled = z.select(['NDVI', 'EVI']).reduceRegions(
        collection=FIELDS_EE,
        reducer=reducer,
        scale=FIELD_SCALE_M,
        tileScale=4,
    )

    def properties_only(feat):
        # Remove geometries from response to keep downloads lightweight.
        f = ee.Feature(feat)
        return ee.Feature(None, f.toDictionary()).set({
            'year': int(metadata['year']),
            'bin_start': metadata['bin_start'],
            'bin_end_exclusive': metadata['bin_end_exclusive'],
            'bin_midpoint': metadata['bin_midpoint'],
            'mid_doy': metadata['mid_doy'],
            'source_scene_count': z.get('scene_count'),
        })

    received = sampled.map(properties_only).getInfo().get('features', [])
    return [f.get('properties', {}) for f in received]


def fetch_bin_one_field_at_a_time(img, metadata):
    """Fallback: one field reduction per request, with no aggregation fanout."""
    z = ee.Image(img).select(['NDVI', 'EVI'])
    rows = []
    for name, feature in zip(['Field_A', 'Field_B', 'Field_C'], field_features):
        f = ee.Feature(feature)
        stats = z.reduceRegion(
            reducer=reducer,
            geometry=f.geometry(),
            scale=FIELD_SCALE_M,
            maxPixels=1e8,
            tileScale=4,
        )
        # Request pixel statistics and source scene count together.
        result = ee.Dictionary({
            'stats': stats,
            'source_scene_count': ee.Image(img).get('scene_count'),
        }).getInfo()
        row = dict(result.get('stats') or {})
        row.update({
            'field_id': name,
            'area_ha': geojson_features[len(rows)]['properties']['area_ha'],
            'year': int(metadata['year']),
            'bin_start': metadata['bin_start'],
            'bin_end_exclusive': metadata['bin_end_exclusive'],
            'bin_midpoint': metadata['bin_midpoint'],
            'mid_doy': metadata['mid_doy'],
            'source_scene_count': result.get('source_scene_count'),
        })
        rows.append(row)
        time.sleep(0.3)
    return rows


metadata_by_year = {
    int(year): [m for m in BIN_METADATA if int(m['year']) == int(year)]
    for year in YEARS
}
completed_bins = get_completed_bins(all_records)

for year in YEARS:
    bins = YEARS_IMAGES[year]
    dates = metadata_by_year[year]
    if len(bins) != len(dates):
        raise ValueError(f'{year}: composite image / date metadata mismatch.')
    for j, (img, meta) in enumerate(zip(bins, dates), start=1):
        key = (int(year), meta['bin_start'])
        if key in completed_bins:
            print(f'{year} bin {j:02d}/{len(bins)}: already saved; skipping.')
            continue

        # A bad response must not be recorded as a complete time bin.
        try:
            try:
                records = fetch_bin_three_fields(img, meta)
            except Exception as exc:
                if 'Too many concurrent aggregations' not in str(exc):
                    raise
                print(f'{year} bin {j:02d}: switching to single-field reductions.')
                records = fetch_bin_one_field_at_a_time(img, meta)
            if (len(records) != 3 or
                    {str(r.get('field_id')) for r in records} != EXPECTED_IDS):
                raise RuntimeError(
                    f'Expected Fields A, B, C for {meta["bin_start"]}; '
                    f'got {[r.get("field_id") for r in records]}'
                )
        except Exception as exc:
            # Save progress before exiting. Re-running only this cell resumes.
            persist_checkpoint(all_records)
            raise RuntimeError(
                f'Extraction stopped at {year} bin {j} ({meta["bin_start"]}). '
                f'{len(get_completed_bins(all_records))} completed bins are saved in '
                f'{CHECKPOINT_PATH}. Rerun this cell to resume.'
            ) from exc

        # Remove any incomplete partial records for this bin, then replace.
        all_records = [r for r in all_records if not (
            str(r.get('bin_start'))[:10] == meta['bin_start'] and
            int(r.get('year', -1)) == int(year)
        )]
        all_records.extend(records)
        persist_checkpoint(all_records)
        completed_bins.add(key)
        print(f'{year} bin {j:02d}/{len(bins)}: saved 3 fields '
              f'({len(completed_bins)} total bins completed).')
        time.sleep(0.3)  # modest pacing; avoids a rapid request burst
    print(f'{year}: extraction finished.')

# Restore the table format expected by subsequent smoothing and export cells.
comp_df = pd.DataFrame(all_records)
needed = ['year', 'field_id', 'area_ha', 'bin_start', 'bin_midpoint', 'mid_doy',
          'NDVI_mean', 'NDVI_count', 'EVI_mean', 'EVI_count', 'source_scene_count']
for col in needed:
    if col not in comp_df:
        comp_df[col] = np.nan
comp_df = comp_df[needed].copy()
for col in ['year', 'area_ha', 'NDVI_mean', 'NDVI_count', 'EVI_mean', 'EVI_count',
            'source_scene_count', 'mid_doy']:
    comp_df[col] = pd.to_numeric(comp_df[col], errors='coerce')
comp_df['date'] = pd.to_datetime(comp_df['bin_midpoint'])
comp_df = comp_df.sort_values(['field_id', 'year', 'date']).reset_index(drop=True)
comp_df['observed'] = comp_df['NDVI_mean'].notna() & (comp_df['NDVI_count'].fillna(0) > 0)

raw_file = DIRS['Tables'] / 'composite_field_timeseries.csv'
comp_df.to_csv(raw_file, index=False)
print(f'Finished: {len(comp_df)} field × time-bin records saved to {raw_file}')
display(comp_df.head(12))

## 7. Short-gap interpolation and Savitzky–Golay smoothing

**Do not interpolate major cloud gaps.** Only interior missing runs of **one or two 10-day bins** are linearly interpolated (configurable). Longer gaps remain `NaN` and are clearly visible in the figures. Savitzky–Golay filtering (quadratic, five-bin window) operates independently within each contiguous data segment of sufficient length; a long missing-data gap is **not bridged**. Original observations remain intact in separate columns.

Use the **smoothed field-mean NDVI** for SOS/POS/EOS, not the map-based pixel peaks.

In [ ]:
def gap_run_max(values):
    flags = np.asarray(values, dtype=bool)
    longest, run = 0, 0
    for flag in flags:
        run = run+1 if flag else 0
        longest = max(longest,run)
    return longest


def fill_short_internal_gaps(series, max_bins=2):
    original = np.asarray(series, dtype=float)
    result = original.copy()
    filled = np.zeros(len(result), dtype=bool)
    i = 0
    while i < len(result):
        if np.isfinite(result[i]):
            i += 1
            continue
        start = i
        while i < len(result) and not np.isfinite(result[i]):
            i += 1
        stop = i
        run_length = stop-start
        if (start>0 and stop<len(result) and run_length<=max_bins
                and np.isfinite(result[start-1]) and np.isfinite(result[stop])):
            result[start:stop] = np.interp(
                np.arange(start,stop), [start-1, stop], [result[start-1],result[stop]])
            filled[start:stop] = True
    return result, filled


def smooth_contiguous(series, window=5, order=2):
    vals = np.asarray(series, dtype=float)
    smoothed = vals.copy()
    i = 0
    while i<len(vals):
        if not np.isfinite(vals[i]):
            i += 1
            continue
        start = i
        while i<len(vals) and np.isfinite(vals[i]):
            i += 1
        stop = i
        n = stop-start
        # Only smooth segments long enough for the selected window.
        if n>=window:
            smoothed[start:stop] = savgol_filter(vals[start:stop], window_length=window,
                                                  polyorder=order, mode='interp')
    return smoothed

smoothed_groups = []
quality_rows = []
for (field_id, year), group in comp_df.groupby(['field_id','year'], sort=True):
    g = group.sort_values('date').copy()
    miss = ~g['observed'].to_numpy()
    quality = {
        'field_id':field_id,'year':int(year), 'total_bins':len(g),
        'observed_bins':int((~miss).sum()),
        'missing_bins':int(miss.sum()),
        'observed_fraction':float((~miss).mean()),
        'longest_missing_run_bins':int(gap_run_max(miss)),
        'total_source_scenes':int(g['source_scene_count'].fillna(0).sum())
    }
    for vi in ['NDVI','EVI']:
        observed = g[f'{vi}_mean'].to_numpy(dtype=float)
        interpolated, filled = fill_short_internal_gaps(observed, MAX_INTERPOLATED_GAP_BINS)
        smoothed = smooth_contiguous(interpolated, SAVGOL_WINDOW_BINS, SAVGOL_POLY_ORDER)
        g[f'{vi}_interpolated'] = interpolated
        g[f'{vi}_smoothed'] = smoothed
        g[f'{vi}_was_interpolated'] = filled
        quality[f'{vi}_interpolated_bins'] = int(filled.sum())
        quality[f'{vi}_remaining_missing_bins'] = int((~np.isfinite(smoothed)).sum())
    quality_rows.append(quality)
    smoothed_groups.append(g)

smooth_df = pd.concat(smoothed_groups,ignore_index=True).sort_values(['field_id','year','date'])
quality_df = pd.DataFrame(quality_rows)
smooth_df.to_csv(DIRS['Tables']/'raw_interpolated_smoothed_timeseries.csv',index=False)
quality_df.to_csv(DIRS['Tables']/'observation_quality_by_field_year.csv',index=False)
print('Saved smoothed data and observation QA. Change MAX_INTERPOLATED_GAP_BINS or SAVGOL_WINDOW_BINS to test sensitivity.')
display(quality_df)

## 8. Phenology: SOS, peak date, EOS, amplitude, and integral

For a field-year series, define baseline as the **10th percentile** of valid smoothed NDVI, peak as the maximum, and amplitude = peak − baseline. **SOS and EOS** are *interpolated dates* when the series crosses the threshold `baseline + fraction × amplitude` on the rising and falling limbs, respectively. Estimate metrics for both **20%** and **30%** threshold fractions.

A field-year is flagged if: fewer than `MIN_VALID_BINS` raw bins; any unresolved **interior** gap (leading/trailing gaps are not extrapolated); seasonal amplitude below `MIN_SEASONAL_AMPLITUDE`; or an index maximum at the season boundary. Threshold crossings may also be undefined when the seasonal curve never crosses the selected level. **No synthetic dates are assigned to these cases.**

**Seasonal NDVI integral** is computed numerically with units **NDVI-days**, and an above-baseline integral is also reported (NDVI-days). These are proxies for greenness, **not biomass measurements**.

In [ ]:
def interpolated_crossing(x0, y0, x1, y1, level):
    if abs(y1-y0) < 1e-12:
        return float(x0)
    return float(x0 + (level-y0)*(x1-x0)/(y1-y0))


def rising_crossing(days, vals, level, peak_pos):
    for i in range(1, peak_pos+1):
        if vals[i-1] < level <= vals[i]:
            return interpolated_crossing(days[i-1], vals[i-1], days[i], vals[i], level)
    return np.nan


def falling_crossing(days, vals, level, peak_pos):
    for i in range(peak_pos+1, len(vals)):
        if vals[i-1] >= level > vals[i]:
            return interpolated_crossing(days[i-1], vals[i-1], days[i], vals[i], level)
    return np.nan


def doy_to_iso(year, doy):
    if not np.isfinite(doy):
        return None
    dt = datetime(int(year),1,1)+timedelta(days=float(doy)-1)
    return dt.strftime('%Y-%m-%d')


def extract_phenology(group, threshold_fraction, info):
    year = int(group['year'].iloc[0]); fid = str(group['field_id'].iloc[0])
    d = group['mid_doy'].to_numpy(dtype=float)
    v = group['NDVI_smoothed'].to_numpy(dtype=float)
    output = {'field_id':fid, 'year':year, 'threshold_fraction':threshold_fraction,
              'SOS_DOY':np.nan,'POS_DOY':np.nan,'EOS_DOY':np.nan,
              'SOS_date':None,'POS_date':None,'EOS_date':None,
              'NDVI_baseline':np.nan,'NDVI_peak':np.nan,'amplitude':np.nan,
              'integrated_NDVI_days':np.nan, 'integrated_above_baseline_days':np.nan,
              'analysis_first_DOY':np.nan,'analysis_last_DOY':np.nan,
              'status':'UNKNOWN'}
    if int(info['observed_bins'])<MIN_VALID_BINS:
        output['status']='INSUFFICIENT_RAW_OBSERVATIONS'; return output
    # Missing bins at the very start/end are permitted, but we never extrapolate
    # them. A gap inside the observed interval invalidates SOS/EOS estimates.
    good = np.isfinite(v)
    if not good.any():
        output['status']='NO_VALID_COMPOSITES'; return output
    first = int(np.argmax(good))
    last = len(v)-1-int(np.argmax(good[::-1]))
    if not np.all(good[first:last+1]):
        output['status']='UNRESOLVED_INTERIOR_GAPS'; return output
    d = d[first:last+1]
    v = v[first:last+1]
    output['analysis_first_DOY'] = float(d[0])
    output['analysis_last_DOY'] = float(d[-1])
    if len(v)<5:
        output['status']='TOO_FEW_VALID_COMPOSITES'; return output
    peak_idx = int(np.argmax(v))
    if peak_idx == 0 or peak_idx == len(v)-1:
        output['status']='PEAK_AT_OBSERVED_BOUNDARY'; return output
    base = float(np.percentile(v,10)); peak = float(v[peak_idx]); amp = peak-base
    output.update({'NDVI_baseline':base,'NDVI_peak':peak,'amplitude':amp})
    if amp < MIN_SEASONAL_AMPLITUDE:
        output['status']='WEAK_SEASONAL_SIGNAL'; return output
    level = base + threshold_fraction * amp
    sos = rising_crossing(d,v,level,peak_idx)
    eos = falling_crossing(d,v,level,peak_idx)
    pos = float(d[peak_idx])
    output.update({
        'SOS_DOY':sos, 'POS_DOY':pos, 'EOS_DOY':eos,
        'SOS_date':doy_to_iso(year,sos), 'POS_date':doy_to_iso(year,pos),
        'EOS_date':doy_to_iso(year,eos),
        'integrated_NDVI_days':float(np.trapezoid(v,d)),
        'integrated_above_baseline_days':float(np.trapezoid(np.clip(v-base,0,None),d)),
        'status':('OK_WITH_EDGE_GAPS' if (first>0 or last<len(group)-1) else 'OK')
                  if np.isfinite(sos) and np.isfinite(eos) else
                 ('SOS_NOT_OBSERVED' if not np.isfinite(sos) else 'EOS_NOT_OBSERVED')
    })
    return output

metric_rows=[]
for (field_id,year), group in smooth_df.groupby(['field_id','year'],sort=True):
    info = quality_df.loc[(quality_df.field_id==field_id)&(quality_df.year==year)].iloc[0]
    for fraction in PHENOLOGY_THRESHOLDS:
        metric_rows.append(extract_phenology(group.sort_values('date'),fraction,info))
metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(DIRS['Tables']/'phenology_metrics_threshold_20_and_30_percent.csv',index=False)
print('Phenology metrics:')
display(metrics_df[['field_id','year','threshold_fraction','SOS_date','POS_date','EOS_date',
                   'amplitude','integrated_NDVI_days','status']])

## 9. Required figures — field dynamics, years, and smoothing sensitivity

Every figure is saved at **300 dpi** in the student's Google Drive. The plotted curves and dates are derived from actual satellite data, not simulated data. Missing values remain gaps. Rows flagged with missing SOS/EOS receive no fabricated dates; complete estimates with missing edge observations are labelled `OK_WITH_EDGE_GAPS`.

In [ ]:
FIELD_COLORS = {'Field_A':'#2166ac','Field_B':'#b2182b','Field_C':'#238b45'}
YEAR_COLORS = {y: plt.get_cmap('tab10')(i % 10) for i,y in enumerate(YEARS)}

# A: compare three fields in each year, for BOTH NDVI and EVI.
for year in YEARS:
    fig, axs=plt.subplots(2,1,figsize=(10,8),sharex=True,constrained_layout=True)
    for i,vi in enumerate(['NDVI','EVI']):
        for fid in FIELD_COLORS:
            g=smooth_df[(smooth_df.year==year)&(smooth_df.field_id==fid)].sort_values('date')
            axs[i].plot(g['date'],g[f'{vi}_smoothed'],lw=2.1,
                        color=FIELD_COLORS[fid],label=fid.replace('_',' '))
            axs[i].scatter(g.loc[g[f'{vi}_mean'].notna(),'date'],
                           g.loc[g[f'{vi}_mean'].notna(),f'{vi}_mean'],
                           color=FIELD_COLORS[fid],s=15,alpha=.55)
        axs[i].set_ylabel(vi+' (unitless)'); axs[i].legend(ncol=3,loc='upper right')
        axs[i].set_title(f'{vi}: 10-day observed field means (dots) and smoothed curves')
    axs[1].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    axs[1].set_xlabel('Month')
    fig.suptitle(f'Sentinel-2 agricultural field growth dynamics — {year}',fontweight='bold')
    path=DIRS['Figures']/f'fields_NDVI_EVI_{year}.png'
    fig.savefig(path,dpi=300,bbox_inches='tight');plt.show();plt.close(fig)

# B: compare different years, separately for each field.
for fid in FIELD_COLORS:
    fig, axs=plt.subplots(2,1,figsize=(10,8),sharex=True,constrained_layout=True)
    for i,vi in enumerate(['NDVI','EVI']):
        for year in YEARS:
            g=smooth_df[(smooth_df.year==year)&(smooth_df.field_id==fid)].sort_values('mid_doy')
            axs[i].plot(g['mid_doy'],g[f'{vi}_smoothed'],lw=2.2,
                        color=YEAR_COLORS[year],label=str(year))
            axs[i].scatter(g.loc[g[f'{vi}_mean'].notna(),'mid_doy'],
                           g.loc[g[f'{vi}_mean'].notna(),f'{vi}_mean'],
                           color=YEAR_COLORS[year],s=13,alpha=.50)
        axs[i].set_ylabel(vi+' (unitless)');axs[i].legend(ncol=len(YEARS))
    axs[1].set_xlabel('Day of year (note leap-year shifts)')
    fig.suptitle(f'Interannual crop greenness — {fid.replace("_"," ")}',fontweight='bold')
    path=DIRS['Figures']/f'interannual_NDVI_EVI_{fid}.png'
    fig.savefig(path,dpi=300,bbox_inches='tight');plt.show();plt.close(fig)

In [ ]:
# C: Explain processing effects with a single annotated example.
example_field = 'Field_A'
example_year = YEARS[-1]
g = smooth_df[(smooth_df.field_id==example_field)&(smooth_df.year==example_year)].sort_values('mid_doy')
fig,ax=plt.subplots(figsize=(10,5.5),constrained_layout=True)
ax.scatter(g['mid_doy'],g['NDVI_mean'],label='Observed 10-day NDVI',color='#555555',s=35,zorder=3)
ax.plot(g['mid_doy'],g['NDVI_interpolated'],label='Short-gap interpolation',lw=1.5,
        linestyle='--',color='#cc7832')
ax.plot(g['mid_doy'],g['NDVI_smoothed'],label='Savitzky–Golay NDVI',lw=2.4,color='#2166ac')
for fraction,color in [(0.20,'#268a47'),(0.30,'#9e3c9b')]:
    row = metrics_df[(metrics_df.field_id==example_field)&
                     (metrics_df.year==example_year)&
                     (np.isclose(metrics_df.threshold_fraction,fraction))].iloc[0]
    for which,style in [('SOS_DOY',':'),('EOS_DOY','--')]:
        x=row[which]
        if pd.notna(x):
            ax.axvline(x,color=color,ls=style,alpha=.8,lw=1.4,
                       label=f'{int(fraction*100)}% {which[:3]}')
ax.set(xlabel='Day of year',ylabel='NDVI (unitless)',
       title=f'Raw observations, smoothing and phenological thresholds: {example_field}, {example_year}')
ax.legend(ncol=3,fontsize=8)
fig.savefig(DIRS['Figures']/'smoothing_and_threshold_sensitivity.png',dpi=300,bbox_inches='tight')
plt.show();plt.close(fig)

# D: Field-year comparison of satellite-estimated timing.
base_metrics = metrics_df[np.isclose(metrics_df.threshold_fraction,0.20)].copy()
base_metrics['field_year'] = base_metrics['field_id'].str.replace('_',' ')+' / '+base_metrics['year'].astype(str)
fig,ax=plt.subplots(figsize=(11,5),constrained_layout=True)
x=np.arange(len(base_metrics));width=.26
for offset, col, name in [(-width,'SOS_DOY','SOS'),(0,'POS_DOY','POS'),(width,'EOS_DOY','EOS')]:
    ax.bar(x+offset, pd.to_numeric(base_metrics[col],errors='coerce'),width=width,label=name)
ax.set_xticks(x,base_metrics['field_year'],rotation=35,ha='right')
ax.set(ylabel='Day of year',title='Satellite-derived phenology estimates (20% threshold)')
ax.legend()
fig.savefig(DIRS['Figures']/'phenology_field_year_comparison.png',dpi=300,bbox_inches='tight')
plt.show();plt.close(fig)
print('All temporal figures saved to:',DIRS['Figures'])

## 10. Spatial metrics: observed composite peak, peak timing, and variability

These maps summarize **pixel-level 10-day composites** and are **not** pixelwise Savitzky–Golay phenology estimates. For each pixel:
- `peak_NDVI` = highest available 10-day median NDVI;
- `peak_DOY` = midpoint day of year of the winning composite;
- `NDVI_p90_minus_p10` = robust within-season range;
- `valid_composite_count` = count of 10-day NDVI observations at that pixel.

Maps are masked where fewer than **six** valid 10-day composites are available. High peak NDVI and an early peak DOY do not prove field condition or yield. PNG quicklooks are *visualizations* and do not substitute for the scientific GeoTIFFs.

In [ ]:
SPATIAL_IMAGES = {}
for year in YEARS:
    ic = YEARS_COLLECTIONS[year]
    peak = ic.qualityMosaic('NDVI').select(['NDVI','DOY'],['peak_NDVI','peak_DOY'])
    pct = ic.select('NDVI').reduce(ee.Reducer.percentile([10,90]))
    amplitude = pct.select('NDVI_p90').subtract(pct.select('NDVI_p10')).rename('NDVI_p90_minus_p10')
    valid_count = ic.select('NDVI').count().rename('valid_composite_count').toFloat()
    safe_mask = valid_count.gte(MIN_PIXEL_COMPOSITES)
    SPATIAL_IMAGES[year] = ee.Image.cat([peak,amplitude,valid_count]).toFloat().updateMask(safe_mask).clip(ROI_EE)
    print(f'{year}: spatial image ready (4 bands).')

# Interactive optional map (lightweight; no forced OpenStreetMap requests).
try:
    preview = geemap.Map(center=[49.60,-98.18],zoom=10,basemap='Esri.WorldImagery')
    preview.centerObject(ROI_EE,12)
    preview.addLayer(SPATIAL_IMAGES[YEARS[-1]].select('peak_NDVI'),
        {'min':0,'max':0.9,'palette':['#5e4fa2','#f7f7f7','#a6d96a','#006837']},
        f'Peak NDVI {YEARS[-1]}')
    preview.addLayer(FIELDS_EE,{'color':'#f0f0f0'},'Field boundaries')
    display(preview)
except Exception as exc:
    print('Interactive preview is optional and unavailable:',exc)

## 11. Save quicklook PNG maps to Google Drive (optional)

The thumbnail endpoint may be unavailable from some Colab accounts or for very large study areas. Failures are recorded instead of stopping the rest of the analysis. Real quantitative map products are exported in the next section as GeoTIFFs.

In [ ]:
QUICKLOOKS = [
    ('peak_NDVI',0,0.9,['#5e4fa2','#f7f7f7','#a6d96a','#006837']),
    ('peak_DOY',120,300,['#440154','#31688e','#35b779','#fde725']),
    ('NDVI_p90_minus_p10',0,0.8,['#fff7bc','#fec44f','#d95f0e','#993404'])
]
quicklook_rows = []
if MAKE_MAP_QUICKLOOKS:
    from PIL import Image as PILImage
    from io import BytesIO
    for year in YEARS:
        for band,vmin,vmax,colors in QUICKLOOKS:
            target=DIRS['Map_Quicklooks']/f'{band}_{year}.png'
            try:
                palette = colors
                img=SPATIAL_IMAGES[year].select(band)
                url=img.getThumbURL({'region':ROI_EE.bounds(maxError=1),
                                     'dimensions':900, 'min':vmin,'max':vmax,
                                     'palette':palette, 'format':'png'})
                with urllib.request.urlopen(url,timeout=90) as response:
                    thumbnail = PILImage.open(BytesIO(response.read())).convert('RGBA')
                fig,ax=plt.subplots(figsize=(8,7),constrained_layout=True)
                ax.imshow(thumbnail)
                ax.axis('off')
                ax.set_title(f'{year} — {band.replace("_"," ")} (satellite quicklook)',fontweight='bold')
                cmap=LinearSegmentedColormap.from_list('index',colors)
                cb=fig.colorbar(ScalarMappable(norm=Normalize(vmin=vmin,vmax=vmax),cmap=cmap),
                                ax=ax,fraction=0.045,pad=0.01)
                cb.set_label('DOY' if band=='peak_DOY' else 'NDVI (unitless)')
                fig.savefig(target,dpi=300,bbox_inches='tight');plt.close(fig)
                quicklook_rows.append({'year':year,'band':band,'status':'SAVED','file':str(target)})
                print('Saved:',target.name)
            except Exception as exc:
                quicklook_rows.append({'year':year,'band':band,'status':'FAILED','reason':str(exc)[:200]})
                print(f'{band}/{year}: thumbnail unavailable ({type(exc).__name__}). GeoTIFF export is independent.')
else:
    print('MAKE_MAP_QUICKLOOKS=False; skipping thumbnails.')
pd.DataFrame(quicklook_rows).to_csv(DIRS['Logs']/'quicklook_status.csv',index=False)

## 12. Export scientific raster maps as GeoTIFFs — asynchronous Earth Engine Drive tasks

The following cell starts up to **three** Earth Engine raster export tasks (one **four-band** GeoTIFF per year). The tasks send results to a **separate folder in the same student's Google Drive** named `P02_<student>_GeoTIFF`. Their completion is **not guaranteed immediately**; monitor their task status in the next cell. Masked pixels have GeoTIFF no-data status through EE export; bands retain their native units as described in Section 10.

> If you only need CSVs and plots, set `START_GEOTIFF_EXPORTS=False` in Section 1 and rerun this cell. Keep field areas reasonably small to avoid unnecessary export volume.

In [ ]:
EXPORT_TASKS=[]
if START_GEOTIFF_EXPORTS:
    for year in YEARS:
        task_name=f'P02_{TAG}_{year}_Spatial_Phenology'
        task=ee.batch.Export.image.toDrive(
            image=SPATIAL_IMAGES[year], description=task_name,
            folder=EXPORT_DRIVE_FOLDER, fileNamePrefix=task_name,
            region=ROI_EE, scale=FIELD_SCALE_M,
            maxPixels=1e10, fileFormat='GeoTIFF')
        task.start()
        EXPORT_TASKS.append({'year':year,'task_name':task_name,'task':task})
        print('Submitted Earth Engine export:',task_name)
else:
    print('GeoTIFF export disabled.')

In [ ]:
def check_export_tasks(records):
    summaries=[]
    for row in records:
        status=row['task'].status()
        summaries.append({'year':row['year'],'task':row['task_name'],
                          'state':status.get('state','UNKNOWN'),
                          'error_message':status.get('error_message',''),
                          'task_id':status.get('id','')})
    result=pd.DataFrame(summaries,columns=['year','task','state','error_message','task_id'])
    result.to_csv(DIRS['Logs']/'geotiff_export_status.csv',index=False)
    return result

display(check_export_tasks(EXPORT_TASKS))
print('Run this cell again later in your active Colab session to refresh task status.')
print('Only a state of COMPLETED means that the GeoTIFF export succeeded.')

## 13. Save configuration, summary, and manifest in the student's Drive

The `Tables/time_bin_extraction_checkpoint.csv` file preserves the per-bin extraction progress and is safe to resume from if Colab is interrupted.

Save the reproducibility metadata, final field-year metric table and the complete inventory of Python-generated results. **The manifest excludes pending GeoTIFF export tasks.** After exports are complete, students can open the separate Drive folder to access the GeoTIFFs. The student-specific output directories prevent one student's results from overwriting another's.

In [ ]:
settings = {
    'student_id':STUDENT_ID, 'student_tag':TAG, 'ee_project':EE_PROJECT_ID,
    'years':YEARS, 'period':'May 1 through October 31',
    'satellite_collection':'COPERNICUS/S2_SR_HARMONIZED',
    'cloud_quality_collection':'GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED',
    'cloud_score_cdf_min':CLOUD_SCORE_THRESHOLD, 'scene_cloud_percent_max':SCENE_CLOUD_LIMIT,
    'scl_accepted_classes':[4,5,6,7], 'spatial_scale_m':FIELD_SCALE_M,
    'composite_days':COMPOSITE_DAYS, 'composite_reducer':'median of index images',
    'interpolation':'linear, interior gaps only, max bins '+str(MAX_INTERPOLATED_GAP_BINS),
    'smoothing':f'Savitzky-Golay order {SAVGOL_POLY_ORDER}; window {SAVGOL_WINDOW_BINS} bins',
    'phenology_baseline':'10th percentile of smoothed NDVI',
    'phenology_thresholds':PHENOLOGY_THRESHOLDS,
    'min_observed_bins':MIN_VALID_BINS, 'min_ndvi_amplitude':MIN_SEASONAL_AMPLITUDE,
    'spatial_peak_method':'max over unsmoothed 10-day median composite NDVI',
    'spatial_amplitude_method':'NDVI p90 minus p10 over 10-day composites',
    'minimum_pixel_composites':MIN_PIXEL_COMPOSITES,
    'student_drive_dir':str(ROOT), 'earth_engine_export_drive_folder':EXPORT_DRIVE_FOLDER,
    'run_saved_utc':datetime.utcnow().isoformat()+'Z'
}
with open(ROOT/'run_configuration.json','w',encoding='utf-8') as f:
    json.dump(settings,f,indent=2)

summary = metrics_df.merge(quality_df,on=['field_id','year'],how='left')
summary.to_csv(DIRS['Tables']/'phenology_and_quality_summary.csv',index=False)
manifest = [{'relative_path':str(p.relative_to(ROOT)), 'size_bytes':p.stat().st_size}
            for p in ROOT.rglob('*') if p.is_file()]
pd.DataFrame(manifest).to_csv(ROOT/'output_manifest.csv',index=False)
print('Saved summary, settings, manifest to your Google Drive:',ROOT)
print('Python files currently recorded:',len(manifest))

## 14. Scientific interpretation and student deliverables

### Deliverables (upload via the course's assignment page; these are already saved in *your* Drive)
1. `selected_fields.geojson`, three field areas and the `run_configuration.json`.
2. Three **field-comparison** figures (one per year) and three **interannual** figures (one per field).
3. `smoothing_and_threshold_sensitivity.png` and `phenology_field_year_comparison.png`.
4. `phenology_and_quality_summary.csv`, the full time-series CSV and observation-quality CSV.
5. GeoTIFF outputs **after Earth Engine task status changes to `COMPLETED`**; PNG map quicklooks when available.
6. A 400–600-word scientific interpretation answering the questions below.

### Questions for students
- Which field reaches maximum NDVI earliest? Does this ranking remain consistent across years?
- Compare NDVI and EVI for the three fields. What do their differences tell you, and what can they **not** tell you about crop identity or yield?
- Do any time series contain missing periods or questionable observations? What happens to apparent season start/end after smoothing?
- For a particular field and year, compare 20% and 30% SOS/EOS estimates. Quantify the shift in **days** and discuss whether the choice changes your biological interpretation.
- How do the **pixelwise observed peak-DOY maps** differ from the **smoothed field-level POS**? Explain the consequences of pixel observation gaps.
- Which results require field management, crop type, climate, or ground phenological observations before drawing causal conclusions?

### Interpretation cautions
- **10-day medians are not daily observations** and a 10-day bin may be empty.
- A green-up threshold crossing is *not* a field-verified emergence/planting date.
- Harvest, re-growth, double cropping and mixed vegetation can create multiple peaks; this notebook uses a **single-main-peak** approach and should not be used to claim a verified crop stage.
- Crop yield, drought, nitrogen status and crop rotation cannot be identified reliably from NDVI/EVI time series alone.
- Earth Engine export submissions remain `READY` or `RUNNING` until completed. Open the separate GeoTIFF Drive folder after completion.

### Data and method references
- Google Earth Engine [Sentinel-2 SR Harmonized collection](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED).
- Google Earth Engine [Cloud Score+](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_CLOUD_SCORE_PLUS_V1_S2_HARMONIZED).
- Google Earth Engine [export to Drive](https://developers.google.com/earth-engine/guides/exporting_images).
- Savitzky, A. & Golay, M. J. E. (1964). Smoothing and Differentiation of Data by Simplified Least Squares Procedures. *Analytical Chemistry*, 36, 1627–1639.
- White, M. A. et al. (2009). Intercomparison, interpretation, and assessment of spring phenology in North America estimated from remote sensing for 1982–2006. *Global Change Biology*, 15, 2335–2359.

**Troubleshooting:** If Earth Engine reports `Too many concurrent aggregations` during extraction, do not re-create the ROI or repeat the setup. The v4 extraction cell processes one 10-day bin at a time and saves each completed bin. Re-run only Section 6 to resume. Avoid running many Earth Engine notebooks simultaneously.
